# Customizing prompts.

Every generative component in RAGU inherits from `RaguGenerativeModule`, which
gives it three methods:

| method | does |
|---|---|
| `get_prompts()` | every prompt *this module* owns, as `{name: RAGUInstruction}` |
| `get_prompt(name)` | one `RAGUInstruction` |
| `update_prompt(name, instruction)` | replace it |

A prompt is a `RAGUInstruction` — a frozen dataclass with four fields:

```python
@dataclass(frozen=True, slots=True)
class RAGUInstruction:
    messages: ChatMessages                          # the Jinja2 templates
    pydantic_model: Type[BaseModel] | Type[str] = str   # output schema
    description: str | None = None
    few_shot_formatter: FewShotFormatter | None = None
```

Two properties matter and are both demonstrated below:

- **Updates are per instance.** `update_prompt` writes into that engine's own dict.
  The global `DEFAULT_PROMPT_TEMPLATES` registry and every other engine are
  untouched, so two engines in the same process can run different prompts.
- **Templates are strict.** The Jinja environment uses `StrictUndefined`, so a
  template referencing a variable the engine does not pass raises at render time
  rather than silently producing an empty string.

Sections 1–5 need **no API keys**. The last three build a small graph and need
`OPENAI_API_KEY`, `LLM_MODEL_NAME` and `EMBEDDER_MODEL_NAME`.

In [ ]:
import dataclasses
import os
from pathlib import Path

from ragu.common.prompts import DEFAULT_PROMPT_TEMPLATES, ChatMessages, SystemMessage, UserMessage
from ragu.common.prompts.messages import render
from ragu.common.prompts.prompt_storage import RAGUInstruction

DATA_DIR = Path("data/en")
QUESTION = "Who created the C programming language and where did they work?"

## 1. What prompts exist at all

`DEFAULT_PROMPT_TEMPLATES` is the registry every module pulls its defaults from.
Reading it needs no setup, so this is the cheapest way to find out what is
customizable.

In [ ]:
print(f"{len(DEFAULT_PROMPT_TEMPLATES)} prompts in the registry\n")
print(f"{'name':<32} {'output schema':<26} roles")
print("-" * 78)
for name, instruction in DEFAULT_PROMPT_TEMPLATES.items():
    schema = getattr(instruction.pydantic_model, "__name__", str(instruction.pydantic_model))
    roles = ", ".join(message.role for message in instruction.messages)
    print(f"{name:<32} {schema:<26} {roles}")

`description` says what each one is for.

In [ ]:
for name in ("local_search", "naive_search", "global_search", "artifact_extraction"):
    print(f"{name:<22} {DEFAULT_PROMPT_TEMPLATES[name].description}")

## 2. Which prompts a module owns

A module only registers the prompts it actually uses, so `get_prompts()` is the
authoritative answer for a given engine — narrower than the full registry.

From the source:

| component | prompts |
|---|---|
| `LocalSearchEngine` | `local_search` |
| `NaiveSearchEngine` | `naive_search` |
| `GlobalSearchEngine` | `global_search_context`, `global_search` |
| `MixSearchEngine` | `mix_search_context`, `mix_search` |
| `QueryPlanEngine` | `query_decomposition`, `query_rewrite` |
| `ArtifactsExtractorLLM` | `artifact_extraction`, `artifact_validation` |
| `EntitySummarizer` | `entity_summarizer`, `cluster_summarize` |
| `RelationSummarizer` | `relation_summarizer` |
| `CommunitySummarizer` | `community_report` |

Once an engine exists, ask it directly rather than trusting a table:

```python
engine.get_prompts().keys()   # -> dict_keys(['local_search'])
```

## 3. Reading a prompt

The defaults for `local_search` and `naive_search` are deliberately near-identical:
same instructions, same three variables. What differs is the *context* each engine
assembles and hands to `{{ context }}`.

In [ ]:
for name in ("local_search", "naive_search"):
    instruction = DEFAULT_PROMPT_TEMPLATES[name]
    print("=" * 78)
    print(f"{name}   (schema: {instruction.pydantic_model.__name__})")
    print("=" * 78)
    for message in instruction.messages:
        print(f"--- [{message.role}] ---")
        print(message.content.strip())
    print()

## 4. The variable contract

This is the part to get right: a replacement template may only use the variables
the engine actually passes. Both search prompts are rendered with exactly three.

| prompt | variables | passed by |
|---|---|---|
| `local_search` | `query`, `context`, `language` | `LocalSearchEngine._render_answer_messages` |
| `naive_search` | `query`, `context`, `language` | `NaiveSearchEngine._render_answer_messages` |

`context` arrives already rendered to text and already truncated to
`max_context_length`, so a custom template receives a string, not objects.

Referencing anything else fails loudly:

In [ ]:
broken = ChatMessages.from_messages([UserMessage("{{ query }} / {{ nonexistent_variable }}")])
try:
    render(broken, query="test", context="ctx", language="english")
except Exception as error:
    print(f"{type(error).__name__}: {error}")

## 5. Writing a replacement

`RAGUInstruction` is frozen, so build a new one. Two ways, and which is right
depends on how many of the four fields you are actually changing:

- **`dataclasses.replace(default, messages=...)`** keeps everything you do not
  name. Use it when the prompt carries something worth inheriting.
- **`RAGUInstruction(messages=..., ...)`** starts from the constructor defaults
  (`pydantic_model=str`, `description=None`, `few_shot_formatter=None`). Use it when
  you are replacing the meaningful fields anyway.

For `local_search` and `naive_search` specifically the two forms are nearly
equivalent: their `pydantic_model` is already `str` and their `few_shot_formatter`
is already `None`, so `replace` only inherits the `description`. It becomes
genuinely load-bearing for the eight prompts with a real output schema and the two
that carry a few-shot formatter — see the gotchas at the end.

Here only the messages change, and inheriting the schema is the point, so `replace`
it is. The description is overridden because the default one no longer describes
what this prompt does.

In [ ]:
TERSE_LOCAL = ChatMessages.from_messages([
    SystemMessage(
        "You are a precise research assistant working from a knowledge graph. "
        "The context contains entities, the relations between them, and the source "
        "text they were extracted from. Prefer the relations when they answer the "
        "question — they are the part a plain vector search cannot give you."
    ),
    UserMessage(
        "Question: {{ query }}\n\n"
        "Context:\n{{ context }}\n\n"
        "Answer in {{ language }}. Rules:\n"
        "1. At most three sentences.\n"
        "2. Name the entities you relied on, in brackets, at the end.\n"
        "3. If the context does not contain the answer, say exactly "
        "'Not supported by the retrieved context.' and stop."
    ),
])

custom_local = dataclasses.replace(
    DEFAULT_PROMPT_TEMPLATES["local_search"],
    messages=TERSE_LOCAL,
    description="Terse local-search answers with explicit entity attribution.",
)

print(f"schema inherited: {custom_local.pydantic_model}")
print(f"roles: {[message.role for message in custom_local.messages]}")

Render it before spending a token on it. `render()` is the same function the engine
calls internally, so what you see here is what the model will see.

In [ ]:
preview = render(
    custom_local.messages,
    query=QUESTION,
    context="**Entities**\nDennis Ritchie, PERSON, Created C.\n**Relations**\nDennis Ritchie, WORKS_AS, Bell Labs",
    language="english",
)[0]

for message in preview.to_openai():
    print(f"--- [{message['role']}] ---")
    print(message["content"])

## 6. Applying it

From here on a graph is needed. Everything above ran for free.

In [ ]:
from ragu import (
    ArtifactsExtractorLLM,
    BuilderArguments,
    KnowledgeGraph,
    LocalSearchEngine,
    NaiveSearchEngine,
    Settings,
    SimpleChunker,
)
from ragu.models.embedder import EmbedderOpenAI
from ragu.models.llm import LLMOpenAI
from ragu.models.openai import CachedAsyncOpenAI
from ragu.search_engine.local_search import LocalParams
from ragu.search_engine.naive_search import NaiveSearchParams
from ragu.utils.ragu_utils import read_text_from_files


Settings.language = "english"
Settings.storage_folder = "ragu_working_dir/custom_prompts_example"

client = CachedAsyncOpenAI(
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
    rate_max_simultaneous=10,
    rate_max_per_minute=100,
)
llm = LLMOpenAI(client=client, model_name=os.environ["LLM_MODEL_NAME"])
embedder = EmbedderOpenAI(client=client, model_name=os.environ["EMBEDDER_MODEL_NAME"])
await embedder.initialize()

knowledge_graph = KnowledgeGraph(
    llm=llm,
    embedder=embedder,
    chunker=SimpleChunker(max_chunk_size=1000),
    artifact_extractor=ArtifactsExtractorLLM(llm=llm, embedder=embedder),
    builder_settings=BuilderArguments(),
)
print(read_text_from_files(DATA_DIR)[:1])
await knowledge_graph.build_from_docs(read_text_from_files(DATA_DIR)[:1])

### Local search, before and after

Two engines over the same graph so the retrieval is identical and only the prompt
differs.

In [ ]:
default_local = LocalSearchEngine(llm=llm, knowledge_graph=knowledge_graph, embedder=embedder)
tuned_local = LocalSearchEngine(llm=llm, knowledge_graph=knowledge_graph, embedder=embedder)
tuned_local.update_prompt("local_search", custom_local)

params = LocalParams(top_k=10)

print("=== default local_search ===")
print((await default_local.query(QUESTION, params)).response)
print("\n=== custom local_search ===")
print((await tuned_local.query(QUESTION, params)).response)

### Updates are scoped to the instance

`update_prompt` mutates only that engine's dict. Nothing leaks into the registry or
into the other engine — which is what lets an A/B comparison like the one above
work at all.

In [ ]:
print(f"registry untouched:      "
      f"{DEFAULT_PROMPT_TEMPLATES['local_search'] is not tuned_local.get_prompt('local_search')}")
print(f"other engine untouched:  "
      f"{default_local.get_prompt('local_search') is DEFAULT_PROMPT_TEMPLATES['local_search']}")

### Naive search

Same mechanism, different job. `naive_search` sees raw chunks with similarity
scores and no graph structure, so a prompt that tells the model to cite chunk
numbers fits it better than one that talks about entities.

In [ ]:
QUOTING_NAIVE = ChatMessages.from_messages([
    SystemMessage(
        "You answer strictly from retrieved text passages. Each passage is numbered "
        "and carries a similarity score. Higher scores are not automatically more "
        "correct — read them all."
    ),
    UserMessage(
        "Question: {{ query }}\n\n"
        "Passages:\n{{ context }}\n\n"
        "Answer in {{ language }}. Quote the exact sentence that supports your "
        "answer, then give the answer in one line. If no passage supports it, say "
        "'No supporting passage.'"
    ),
])

default_naive = NaiveSearchEngine(
    llm=llm, 
    knowledge_graph=knowledge_graph, 
    embedder=embedder
)
tuned_naive = NaiveSearchEngine(
    llm=llm, 
    knowledge_graph=knowledge_graph, 
    embedder=embedder
)
tuned_naive.update_prompt(
    "naive_search",
    RAGUInstruction(messages=QUOTING_NAIVE),
)

naive_params = NaiveSearchParams(top_k=10)

print("=== default naive_search ===")
print((await default_naive.query(QUESTION, naive_params)).response)
print("\n=== custom naive_search ===")
print((await tuned_naive.query(QUESTION, naive_params)).response)

## 7. Changing the output schema

`pydantic_model` is passed straight through to the LLM call:

```python
answers = await self.llm.batch_chat_completion(
    conversations,
    output_schema=instruction.pydantic_model or str,
)
```

So swapping it turns a search engine into a structured extractor.
`SearchEngineResponse.response` is typed `str | BaseModel` and its `__str__` dumps
the model as indented JSON.

In [ ]:
from pydantic import BaseModel, Field


class CitedAnswer(BaseModel):
    """
    Structured answer with explicit grounding.
    """

    answer: str = Field(description="The answer in one or two sentences")
    entities_used: list[str] = Field(description="Names of entities the answer relied on")
    confident: bool = Field(description="False if the context did not really support the answer")


structured_local = LocalSearchEngine(
    llm=llm, 
    knowledge_graph=knowledge_graph,
    embedder=embedder
)

structured_local.update_prompt(
    "local_search",
    RAGUInstruction(
        messages=ChatMessages.from_messages([
            UserMessage(
                "Question: {{ query }}\n\nContext:\n{{ context }}\n\n"
                "Answer in {{ language }} and fill every field. Set confident=false "
                "if the context does not really support the answer."
            ),
        ]),
        pydantic_model=CitedAnswer,
        description="Local search returning a structured, self-scored answer.",
    ),
)

response = await structured_local.query(QUESTION, params)
print(f"type: {type(response.response).__name__}")
print(f"answer:        {response.response.answer}")
print(f"entities_used: {response.response.entities_used}")
print(f"confident:     {response.response.confident}")

## 8. Gotchas

**Streaming ignores the schema.** `stream_query` calls `stream_chat_completion`,
which never receives `output_schema` — a Pydantic model cannot be validated before
the response is complete. An engine configured like the one above will still stream
plain text, so keep structured and streaming engines separate.

In [ ]:
chunks = []
async for event in structured_local.stream_query(QUESTION, params):
    chunks.append(event.delta)
print(f"streamed {len(chunks)} text deltas, type {type(''.join(chunks)).__name__} — not CitedAnswer")

**Other modules may alidate their schema.** 

Search prompts are free-form, but
components that parse structured output guard themselves with
`require_prompt_schema`. Replacing `entity_summarizer` with a prompt whose
`pydantic_model` is not `EntityDescriptionModel` raises:

```
ValueError: Prompt 'entity_summarizer' must use output schema
EntityDescriptionModel, got MyModel. The prompt template was likely replaced
with an incompatible schema via update_prompt().
```

So for those, change `messages` and keep `pydantic_model` — exactly what
`dataclasses.replace` does by default.

**`language` is a variable, not a setting.** The engine passes its own `language`
attribute (from `Settings.language` or the constructor override). Dropping
`{{ language }}` from a template does not break rendering — it just stops telling
the model what language to answer in.

**Few-shot formatters are prompt-specific.** `artifact_extraction` and
`artifact_validation` carry a `few_shot_formatter` used by the ICL manager.
`dataclasses.replace` preserves it; constructing a bare `RAGUInstruction` drops it
and silently disables in-context examples for that prompt.

In [ ]:
await knowledge_graph.index.close()